# DD2360 Assignment 2
Runtime → Change runtime type → **T4 GPU** before running. All build output goes to `build/`.

In [1]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
REPO = "tattack99/DD2360-assignment-2"
REPO_DIR = "/content/repo"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        from google.colab import userdata
        token = userdata.get("GH-token")
        !git clone -q https://{token}@github.com/{REPO}.git {REPO_DIR}
    %cd {REPO_DIR}
    !git pull -q

/content/repo


In [2]:
# Check that a GPU is attached and nvcc is available
!nvidia-smi
!nvcc --version

Wed Nov 18 09:42:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Q1 - 1D Convolution

In [10]:
# Quick check: both kernels should print PASS
!make run

make: *** No rule to make target 'run'.  Stop.


In [ ]:
# Q1.3 tile size: run every tile size on the largest input, compare the tiled kernel times
!make tilesweep ARGS=16384

In [ ]:
# Q1.3 screenshot: N = 1024 ... 16384 with the chosen tile size
TILE = 256
!make sweep TILE={TILE}

In [ ]:
# Q1.4 Nsight Systems: where the time goes
!make profile TILE={TILE} ARGS=16384

In [ ]:
# Q1.4 Nsight Compute: occupancy, shared memory, memory throughput
!make ncu TILE={TILE} ARGS=16384

## Q2 - Tiled Matrix Multiplication

In [4]:
%cd /content/repo/assignment
!ls

/content/repo/assignment
hw2-convolution.cu  matmul.cu


In [5]:
!nvcc -O2 -arch=native -o matmul matmul.cu

In [6]:
# Q2.2
!./matmul 1024 1024 1024 float 1 8 8 16 16 32 32

data type: float
matrix A: (1024 x 1024)
matrix B: (1024 x 1024)
matrix C: (1024 x 1024)
floating-point operations: 2*M*N*P = 2147483648
basic gemm: 16 x 16 threads per block, 4096 blocks, 1048576 threads launched, 0 exit idle
basic gemm global memory reads: 2*M*N*P = 2147483648
bytes copied host to device: 8388608 (A and B)
bytes copied device to host: 4194304 (C)
Achieved Occupancy / shared memory (one tile per run): make occupancy PROG=matmul ARGS="1024 1024 1024 float 0 8 8"
copy data from host to device: 2.728 ms
CPU reference: computing all 1048576 elements
CPU version: 3222.508 ms

kernel                      time (ms)    check   shared/block         global reads  details
gemm (16x16)                    8.967     PASS              0           2147483648  max abs err 9.15527e-05
tiled_gemm (8x8)               11.130     PASS          512 B            268435456  max abs err 9.15527e-05
tiled_gemm (16x16)              6.916     PASS         2048 B            134217728  max abs err 

In [7]:
# Q2.3
!./matmul 513 8192 1023 float 1 8 8 16 16 32 32

data type: float
matrix A: (513 x 8192)
matrix B: (8192 x 1023)
matrix C: (513 x 1023)
floating-point operations: 2*M*N*P = 8598306816
basic gemm: 16 x 16 threads per block, 2112 blocks, 540672 threads launched, 15873 exit idle
basic gemm global memory reads: 2*M*N*P = 8598306816
bytes copied host to device: 50331648 (A and B)
bytes copied device to host: 2099196 (C)
Achieved Occupancy / shared memory (one tile per run): make occupancy PROG=matmul ARGS="513 8192 1023 float 0 8 8"
copy data from host to device: 10.942 ms
CPU reference: checking 3 rows (3069 elements); a full CPU pass is slower than the GPU here
CPU version: 190.559 ms for 3 rows -> 32585.603 ms estimated for all 513 rows

kernel                      time (ms)    check   shared/block         global reads  details
gemm (16x16)                   26.222     PASS              0           8598306816  max abs err 0.000488281
tiled_gemm (8x8)               31.984     PASS          512 B           1074788352  max abs err 0.00048

In [8]:
# Q2.4 Nsight Systems: 1024^3 and 513x8192x1023 (CPU check off so it is quick)
!nsys profile --stats=true --force-overwrite true -o /tmp/mm_1024 ./matmul 1024 1024 1024 float 0 16 16
!nsys profile --stats=true --force-overwrite true -o /tmp/mm_513  ./matmul 513 8192 1023 float 0 16 16

data type: float
matrix A: (1024 x 1024)
matrix B: (1024 x 1024)
matrix C: (1024 x 1024)
floating-point operations: 2*M*N*P = 2147483648
basic gemm: 16 x 16 threads per block, 4096 blocks, 1048576 threads launched, 0 exit idle
basic gemm global memory reads: 2*M*N*P = 2147483648
bytes copied host to device: 8388608 (A and B)
bytes copied device to host: 4194304 (C)
Achieved Occupancy / shared memory (one tile per run): make occupancy PROG=matmul ARGS="1024 1024 1024 float 0 16 16"
copy data from host to device: 2.059 ms
CPU reference: skipped

kernel                      time (ms)    check   shared/block         global reads  details
gemm (16x16)                    7.481      n/a              0           2147483648  max abs err 0
tiled_gemm (16x16)              7.493      n/a         2048 B            134217728  max abs err 0

CSV,float,1024,1024,1024,0.000,0,7.4808,16,16,7.4930
Generating '/tmp/nsys-report-f4ef.qdstrm'
[1/8] [========================100%] mm_1024.nsys-rep
[2/8] [=====

In [9]:
# Q2.4 Nsight Compute: achieved occupancy + shared memory, one tile size per run
for size in ("1024 1024 1024", "513 8192 1023"):
    for t in (8, 16, 32):
        print(f"===== {size}, tile {t}x{t} =====")
        !ncu --section LaunchStats --section Occupancy -k regex:tiled_gemm -c 1 ./matmul {size} float 0 {t} {t}
    print(f"===== {size}, basic gemm =====")
    !ncu --section LaunchStats --section Occupancy -k regex:^gemm -c 1 ./matmul {size} float 0 16 16

===== 1024 1024 1024, tile 8x8 =====
data type: float
matrix A: (1024 x 1024)
matrix B: (1024 x 1024)
matrix C: (1024 x 1024)
floating-point operations: 2*M*N*P = 2147483648
basic gemm: 16 x 16 threads per block, 4096 blocks, 1048576 threads launched, 0 exit idle
basic gemm global memory reads: 2*M*N*P = 2147483648
bytes copied host to device: 8388608 (A and B)
bytes copied device to host: 4194304 (C)
Achieved Occupancy / shared memory (one tile per run): make occupancy PROG=matmul ARGS="1024 1024 1024 float 0 8 8"
==PROF== Connected to process 3579 (/content/repo/assignment/matmul)
copy data from host to device: 2.093 ms
CPU reference: skipped

kernel                      time (ms)    check   shared/block         global reads  details
gemm (16x16)                    6.772      n/a              0           2147483648  max abs err 0
==PROF== Profiling "tiled_gemm": 0%....50%....100% - 1 pass
tiled_gemm (8x8)               12.012      n/a          512 B            268435456  max abs err 